In [11]:
import re


def extract_times(text):
    """
    Извлекает все временные метки формата HH:MM:SS
    из произвольного текста.
    """

    pattern = r"\b\d{2}:\d{2}:\d{2}\b"

    times = re.findall(pattern, text)

    return times


In [ ]:
raw_text ="""сюда вставить текст"""

In [13]:
text = extract_times(raw_text)

In [14]:
text

['15:33:47',
 '14:56:33',
 '16:29:19',
 '16:23:55',
 '16:20:06',
 '16:15:29',
 '16:10:05',
 '15:48:09',
 '15:40:17',
 '15:31:18',
 '15:23:27',
 '15:14:38',
 '15:08:32',
 '15:06:17',
 '15:00:52',
 '14:52:13']

In [16]:
from datetime import datetime


def parse_time(time_str):
    return datetime.strptime(time_str, "%H:%M:%S")


def format_seconds(seconds):
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    return f"{hours} ч {minutes} мин {secs} сек"


def calculate_work_and_breaks(times, break_threshold_minutes=15):
    # Преобразуем строки времени в datetime
    parsed_times = [parse_time(t) for t in times]

    # Сортируем по времени
    parsed_times.sort()

    sessions = []
    breaks = []

    session_start = parsed_times[0]
    prev_time = parsed_times[0]

    for current_time in parsed_times[1:]:
        diff_seconds = int((current_time - prev_time).total_seconds())

        # Если перерыв больше порога — заканчиваем текущую сессию
        if diff_seconds > break_threshold_minutes * 60:
            sessions.append((session_start, prev_time))

            breaks.append({
                "start": prev_time,
                "end": current_time,
                "duration": diff_seconds
            })

            session_start = current_time

        prev_time = current_time

    # Добавляем последнюю сессию
    sessions.append((session_start, prev_time))

    # Считаем общее рабочее время
    total_work_seconds = sum(
        int((end - start).total_seconds())
        for start, end in sessions
    )

    # Считаем общее время перерывов
    total_break_seconds = sum(
        b["duration"]
        for b in breaks
    )

    # Вывод
    print("Рабочие сессии:")
    for i, (start, end) in enumerate(sessions, 1):
        duration = int((end - start).total_seconds())
        print(
            f"{i}. {start.time()} — {end.time()} "
            f"= {format_seconds(duration)}"
        )

    print("\nПерерывы:")
    for i, b in enumerate(breaks, 1):
        print(
            f"{i}. {b['start'].time()} — {b['end'].time()} "
            f"= {format_seconds(b['duration'])}"
        )

    print("\nИТОГО:")
    print("Работа:", format_seconds(total_work_seconds))
    print("Перерывы:", format_seconds(total_break_seconds))


In [19]:
calculate_work_and_breaks(text, break_threshold_minutes=15)

Рабочие сессии:
1. 14:52:13 — 15:48:09 = 0 ч 55 мин 56 сек
2. 16:10:05 — 16:29:19 = 0 ч 19 мин 14 сек

Перерывы:
1. 15:48:09 — 16:10:05 = 0 ч 21 мин 56 сек

ИТОГО:
Работа: 1 ч 15 мин 10 сек
Перерывы: 0 ч 21 мин 56 сек


In [1]:
raw_text ="""[Global] PQ QCS
80
Время выполнения и комментарий
Бонусы
Стоимость
13:21:31
20
13:02:13
20
11:15:31
20
09:56:49
20

[Global] PQ Task
2 877
Время выполнения и комментарий
Бонусы
Стоимость
13:41:52
126
13:35:36
126
13:29:42
126
13:26:25
126
13:21:02
126
13:15:36
126
13:12:25
126
13:00:08
105
12:52:37
105
12:47:01
105
12:44:35
105
12:40:39
105
11:36:50
105
11:31:41
105
11:28:24
105
11:26:34
105
11:20:01
105
11:14:24
105
11:07:25
105
10:59:07
105
10:53:35
105
10:45:02
105
10:27:59
105
10:20:22
105
10:13:41
105
10:04:56
105

[Global] PQ Appeal
15
Время выполнения и комментарий
Бонусы
Стоимость
09:54:04
5
09:49:43
5
09:38:23
5"""

In [4]:
text = extract_times(raw_text)

In [9]:
calculate_work_and_breaks(text, break_threshold_minutes=15)

Рабочие сессии:
1. 09:38:23 — 10:27:59 = 0 ч 49 мин 36 сек
2. 10:45:02 — 11:36:50 = 0 ч 51 мин 48 сек
3. 12:40:39 — 13:41:52 = 1 ч 1 мин 13 сек

Перерывы:
1. 10:27:59 — 10:45:02 = 0 ч 17 мин 3 сек
2. 11:36:50 — 12:40:39 = 1 ч 3 мин 49 сек

ИТОГО:
Работа: 2 ч 42 мин 37 сек
Перерывы: 1 ч 20 мин 52 сек
